# torch.profiler — find where your model is slow

The profiler tells you exactly which operations eat time and memory.
Without this, optimization is guesswork.

## Why this matters for LLMs

```
You:   "My model is slow"
Wrong: "Let me try random optimizations"
Right: "Let me profile it, find the bottleneck, fix THAT"
```

Two types of bottlenecks:
- **Compute-bound**: GPU ALUs are maxed out (rare for LLMs)
- **Memory-bound**: GPU is waiting for data transfers (common for LLMs)

The profiler shows you which one you have.

## Java analogy
This is like JVM profiling with VisualVM / YourKit — but for GPU ops instead of Java methods.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, record_function, ProfilerActivity
import math

# --- Simple model to profile ---
class SimpleTransformerBlock(nn.Module):
    def __init__(self, d_model=256, num_heads=4, ffn_dim=512):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn  = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        with record_function('ATTENTION'):       # label for profiler
            h = self.norm1(x)
            h, _ = self.attn(h, h, h)
            x = x + h
        with record_function('FFN'):             # label for profiler
            x = x + self.ffn(self.norm2(x))
        return x

model = SimpleTransformerBlock()
x = torch.randn(4, 128, 256)   # [batch, seq, dim]

print('Model ready for profiling')

In [ ]:
# --- Basic profiling on CPU ---
# On GPU: change ProfilerActivity.CPU to ProfilerActivity.CUDA

with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,         # log tensor shapes
    profile_memory=True,        # track memory allocation
    with_stack=True,            # capture Python call stack
) as prof:
    for _ in range(10):         # run multiple times for stable measurements
        out = model(x)

# Print top operations by CPU time
print(prof.key_averages().table(
    sort_by='cpu_time_total',
    row_limit=15
))

In [ ]:
# --- See our custom labels (ATTENTION vs FFN) ---
print('\n=== Time split by component ===')
print(prof.key_averages().table(
    sort_by='cpu_time_total',
    row_limit=5
))

# Group by input shape — shows which tensor sizes are expensive
print('\n=== Grouped by input shape ===')
print(prof.key_averages(group_by_input_shape=True).table(
    sort_by='cpu_time_total',
    row_limit=10
))

In [ ]:
# --- Memory profiling ---
print('=== Memory allocation by operation ===')
print(prof.key_averages().table(
    sort_by='self_cpu_memory_usage',
    row_limit=10
))

## Profiling on GPU (Colab)

```python
# On Colab with GPU runtime:
device = 'cuda'
model  = SimpleTransformerBlock().to(device)
x      = torch.randn(4, 128, 256, device=device)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for _ in range(10):
        out = model(x)
    torch.cuda.synchronize()   # IMPORTANT: GPU ops are async, must sync before reading results

print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))
```

**Key insight:** `cuda_time_total` vs `cpu_time_total` — if CPU time >> CUDA time, your GPU is idle waiting for Python. If CUDA time is high, that's where to optimize.

In [ ]:
# --- Export to Chrome trace (visual timeline) ---
# Open chrome://tracing in Chrome and load this file

prof.export_chrome_trace('trace.json')
print('Exported trace.json — open in chrome://tracing for visual timeline')

# On Colab: use TensorBoard integration instead
# with profile(
#     on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profiler'),
# ) as prof:
#     ...
# Then: %load_ext tensorboard  /  %tensorboard --logdir ./log/profiler

## Sequence length scaling — the O(N^2) problem

Profile attention at different sequence lengths to see why Flash Attention matters.

In [ ]:
import time

print(f'{"Seq Len":>8} | {"Time (ms)":>10} | {"Memory (MB)":>12}')
print('-' * 40)

for seq_len in [64, 128, 256, 512, 1024, 2048]:
    x = torch.randn(1, seq_len, 256)

    with profile(activities=[ProfilerActivity.CPU], profile_memory=True) as prof:
        for _ in range(5):
            _ = model(x)

    stats   = prof.key_averages()
    total_t = sum(s.cpu_time_total for s in stats) / 5 / 1000   # avg ms
    total_m = sum(s.self_cpu_memory_usage for s in stats) / 5 / 1e6  # avg MB

    print(f'{seq_len:>8} | {total_t:>10.1f} | {total_m:>12.1f}')

print('\nNotice: time grows ~quadratically with sequence length (O(N^2) attention)')

## Takeaways

| Tool | What it tells you |
|---|---|
| `prof.key_averages().table(sort_by='cpu_time_total')` | Which ops are slowest |
| `sort_by='self_cpu_memory_usage'` | Which ops allocate most memory |
| `record_function('LABEL')` | Custom labels for your code sections |
| `export_chrome_trace()` | Visual timeline in Chrome |
| `ProfilerActivity.CUDA` | GPU-specific timings (Colab) |

**Rule:** profile first, optimize second. Never guess.